# JARVIS-DFT Skutterudite Thermoelectric Data -> SISSO-Ready Dataset

Covers **both**:
- **Binary skutterudites** (CoSb3-type): 2 elements, 1:3 ratio, space group Im-3 (#204)
- **Filled skutterudites** (LaFe4Sb12-type): 3 elements, 1:4:12 ratio, space group Im-3 (#204)

**Target (y):** Seebeck coefficient (n- and p-type combined via long format)
**Input variables (X):** carrier_sign, mean electronegativity, mean valence electron count, mean covalent radius, electronegativity range, band gap, mean melting point, mean unfilled orbitals, ehull

Seebeck reference point in JARVIS: 600 K, 10^20 cm^-3 doping. n-type negative, p-type positive.

Run top to bottom in Google Colab (this notebook is self-contained -- no need to run anything else first).

## 1. Install dependencies

In [ ]:
!pip install jarvis-tools pandas matminer pymatgen -q

## 2. Download JARVIS dft_3d

In [ ]:
import pandas as pd
from jarvis.db.figshare import data as jdata

d = jdata("dft_3d")
df = pd.DataFrame(d)
print(f"Total entries: {len(df)}")

## 3. Elements + TE-data flag

In [ ]:
def get_elements(row):
    try:
        return set(row["atoms"]["elements"])
    except Exception:
        return set()

df["elements"] = df.apply(get_elements, axis=1)
df["n_elements"] = df["elements"].apply(len)

def has_te_data(row):
    return (row.get("n-Seebeck", "na") not in ("na", None)) or (row.get("p-Seebeck", "na") not in ("na", None))

df["has_te_data"] = df.apply(has_te_data, axis=1)

keep_cols = ["jid", "formula", "spg_number", "n-Seebeck", "p-Seebeck", "optb88vdw_bandgap", "ehull"]
keep_cols = [c for c in keep_cols if c in df.columns]

## 4. Filter for Skutterudites (binary + filled)
- Binary: 2 elements, 1:3 stoichiometry ratio, space group 204 (Im-3)
- Filled: 3 elements, 1:4:12 stoichiometry ratio, space group 204 (Im-3)

In [ ]:
def is_binary_skutterudite(row):
    if row["n_elements"] != 2:
        return False
    try:
        if int(row.get("spg_number", -1)) != 204:
            return False
    except (ValueError, TypeError):
        return False
    try:
        comp = row["atoms"]["elements"]
        counts = sorted([comp.count(e) for e in set(comp)])
        return counts[1] == 3 * counts[0]
    except Exception:
        return False

def is_filled_skutterudite(row):
    if row["n_elements"] != 3:
        return False
    try:
        if int(row.get("spg_number", -1)) != 204:
            return False
    except (ValueError, TypeError):
        return False
    try:
        comp = row["atoms"]["elements"]
        counts = sorted([comp.count(e) for e in set(comp)])
        return counts[1] == 4 * counts[0] and counts[2] == 12 * counts[0]
    except Exception:
        return False

binary_mask = df.apply(is_binary_skutterudite, axis=1)
filled_mask = df.apply(is_filled_skutterudite, axis=1)

print(f"Binary skutterudite structures (total):        {binary_mask.sum()}")
print(f"Binary skutterudite structures with TE data:   {(binary_mask & df['has_te_data']).sum()}")
print(f"Filled skutterudite structures (total):        {filled_mask.sum()}")
print(f"Filled skutterudite structures with TE data:   {(filled_mask & df['has_te_data']).sum()}")

skutt_mask = (binary_mask | filled_mask) & df["has_te_data"]
df_skutt = df.loc[skutt_mask, keep_cols].reset_index(drop=True)
print(f"Total skutterudite entries with TE data: {len(df_skutt)}")
df_skutt.head(10)

## 5. Reshape to long format (n/p -> carrier_sign)

In [ ]:
df_long = pd.melt(
    df_skutt,
    id_vars=[c for c in df_skutt.columns if c not in ["n-Seebeck", "p-Seebeck"]],
    value_vars=["n-Seebeck", "p-Seebeck"],
    var_name="doping_type",
    value_name="Seebeck"
)
df_long["doping_type"] = df_long["doping_type"].str.replace("-Seebeck", "")
df_long = df_long[df_long["Seebeck"] != "na"].copy()
df_long["Seebeck"] = df_long["Seebeck"].astype(float)
df_long["carrier_sign"] = df_long["doping_type"].map({"n": -1, "p": 1})

print(f"Long-format entries: {len(df_long)}")

## 6. Compute Magpie elemental descriptors, keep the agreed variable set

In [ ]:
from matminer.featurizers.composition import ElementProperty
from pymatgen.core import Composition

featurizer = ElementProperty.from_preset("magpie")
feature_labels = featurizer.feature_labels()

def safe_composition(formula):
    try:
        return Composition(formula)
    except Exception:
        return None

df_long["composition"] = df_long["formula"].apply(safe_composition)
df_long = df_long[df_long["composition"].notna()].reset_index(drop=True)

def featurize_row(comp):
    try:
        return featurizer.featurize(comp)
    except Exception:
        return [None] * len(feature_labels)

features = df_long["composition"].apply(featurize_row)
features_df = pd.DataFrame(features.tolist(), columns=feature_labels, index=df_long.index)
df_full = pd.concat([df_long.drop(columns=["composition"]), features_df], axis=1)
df_full = df_full.dropna(subset=feature_labels)

selected = [
    "MagpieData mean Electronegativity",
    "MagpieData mean NValence",
    "MagpieData mean CovalentRadius",
    "MagpieData range Electronegativity",
    "MagpieData mean MeltingT",
    "MagpieData mean NUnfilled",
]
selected = [c for c in selected if c in df_full.columns]

final_cols = ["jid", "formula", "doping_type", "carrier_sign",
              "optb88vdw_bandgap", "ehull", "Seebeck"] + selected
final_cols = [c for c in final_cols if c in df_full.columns]

df_sisso = df_full[final_cols].dropna().reset_index(drop=True)
print(f"Final SISSO-ready rows: {len(df_sisso)}")
print(f"Input variables: {[c for c in df_sisso.columns if c not in ['jid','formula','doping_type','Seebeck']]}")
df_sisso.head()

## 7. Save CSV

In [ ]:
df_sisso.to_csv("skutterudite_SISSO_ready.csv", index=False)
print("Saved: skutterudite_SISSO_ready.csv")

## 8. (Optional) Download

In [ ]:
from google.colab import files
files.download("skutterudite_SISSO_ready.csv")